# **Building Production-Grade Data Agents on GCP**

## Notebook 2 — Give Your Agent Enterprise Expertise

In this lab, you will turn the single inventory agent from Notebook 1 into three bounded domain specialists: **Inventory**, **Supplier** and **Fulfilment**. You will work with the semantic catalogue, inspect reusable Agent Skills, give each specialist only its permitted BigQuery tools, and test specialist selection with a deterministic router.

**Facilitated time: 35 minutes.** This notebook is designed to run independently after the pre-workshop environment setup. It does not depend on variables created in Notebook 1.

**Colab-only workflow:** run the cells in order. Edit only the Participant configuration cell. All data access is read-only.

In [ ]:
# Bootstrap: clone the workshop repository and install its dependencies.
# ORGANISER: replace the repository URL once before publishing the workshop.
from pathlib import Path
import subprocess

REPOSITORY_URL = "https://github.com/REPLACE_WITH_GITHUB_OWNER/enterprise-data-agents-gdg-essex.git"
REPOSITORY_REF = "main"
REPO_DIR = Path("/content/enterprise-data-agents-gdg-essex")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
         REPOSITORY_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("Workshop repository already cloned in this runtime.")

%cd /content/enterprise-data-agents-gdg-essex
%pip install -q -r requirements-agent.txt
print("Repository ready. Next, edit the Participant configuration cell.")

In [ ]:
# PARTICIPANT CONFIGURATION — edit the values below, then run this cell.
# Your organiser will provide the source project and source dataset identifiers.
# Do not add passwords, API keys or other credentials here.

PARTICIPANT_CONFIG = {
    "google_cloud": {
        "project_id": "REPLACE_WITH_YOUR_GCP_PROJECT_ID",
        "bigquery_location": "US",
        "dataset_id": "essex_workshop",
    },
    "vertex_ai": {
        "model_id": "gemini-3.8-flash",
        "location": "global",
    },
    "source_data": {
        "project_id": "REPLACE_WITH_WORKSHOP_SOURCE_PROJECT_ID",
        "dataset_id": "REPLACE_WITH_WORKSHOP_SOURCE_DATASET_ID",
    },
    "repository": {
        "url": REPOSITORY_URL,
        "branch_or_tag": REPOSITORY_REF,
    },
}


### What this configuration controls

The configuration separates three things that are easy to confuse:

- **Your project** is where the workshop dataset and views are prepared.
- **BigQuery location** is the data processing location (`US` for this workshop dataset).
- **Vertex AI location** is where the model request is sent (`global` in this example).

The source project and dataset are supplied by the organiser. The setup module prepares the workshop dataset in your project. This cell writes a runtime-local YAML file; it does not change the public GitHub repository.

In [ ]:
# Review the non-secret settings that will be used by this runtime.
# This is a quick configuration sanity check before cloud resources are prepared.
for section, values in PARTICIPANT_CONFIG.items():
    print(f"\n{section}")
    for key, value in values.items():
        print(f"  {key}: {value}")

required_values = [
    PARTICIPANT_CONFIG["google_cloud"]["project_id"],
    PARTICIPANT_CONFIG["source_data"]["project_id"],
    PARTICIPANT_CONFIG["source_data"]["dataset_id"],
]
if any("REPLACE_WITH" in str(value) for value in required_values):
    raise ValueError("Replace the organiser-provided project and dataset placeholders before continuing.")
print("\nConfiguration placeholders check passed.")

# Persist the configuration in this temporary Colab clone and initialise cloud access.
import yaml
config_path = REPO_DIR / "config" / "workshop_config.yaml"
config_path.write_text(
    yaml.safe_dump(PARTICIPANT_CONFIG, sort_keys=False),
    encoding="utf-8",
)

from setup.workshop_setup import initialise_workshop
workshop = initialise_workshop(config_path=config_path, repo_dir=REPO_DIR)

bq = workshop.bigquery
PROJECT_ID = workshop.project_id
DATASET_ID = workshop.dataset_id
BIGQUERY_LOCATION = workshop.location
MODEL_ID = workshop.model_id
MODEL_LOCATION = workshop.model_location

print("Workshop setup complete.")
print(f"Project: {PROJECT_ID} | Dataset: {DATASET_ID} | BigQuery: {BIGQUERY_LOCATION}")
print(f"Gemini: {MODEL_ID} | Vertex AI location: {MODEL_LOCATION}")


## The 35-minute lab plan

| Time | Activity | Outcome |
|---|---|---|
| 0–5 min | Reconnect and inspect the semantic catalogue | Shared business language |
| 5–10 min | Read the three Agent Skills | Reusable domain procedures |
| 10–15 min | Build and test scoped tool surfaces | Least-privilege tool access |
| 15–25 min | Construct three ADK specialists | Distinct agent responsibilities |
| 25–30 min | Run specialist questions | Evidence-backed domain answers |
| 30–35 min | Exercise deterministic routing and review | Specialist selection and boundaries |

The aim is not to make three agents that all know everything. The aim is to make three agents whose **responsibilities, context and permitted tools are deliberately different**.

## 1. Reconnect to the workshop data and shared code

The setup module authenticates with your Google account, configures Vertex AI and prepares the participant BigQuery dataset. It is safe to run this notebook without running Notebook 1 first. The module is shared across the practical notebooks so setup behaviour stays consistent.

In [ ]:
from pathlib import Path
from setup.workshop_setup import initialise_workshop

REPO_DIR = Path('/content/enterprise-data-agents-gdg-essex')
config_path = REPO_DIR / 'config' / 'workshop_config.yaml'
workshop = initialise_workshop(config_path=config_path, repo_dir=REPO_DIR)

bq = workshop.bigquery
PROJECT_ID = workshop.project_id
DATASET_ID = workshop.dataset_id
BIGQUERY_LOCATION = workshop.location
MODEL_ID = workshop.model_id
MODEL_LOCATION = workshop.model_location

print(f"Project: {PROJECT_ID}")
print(f"Dataset: {DATASET_ID} ({BIGQUERY_LOCATION})")
print(f"Model: {MODEL_ID} ({MODEL_LOCATION})")

### Why repeat setup in each notebook?

Colab runtimes are temporary. Participants may open Notebook 2 directly, use a fresh runtime, or lose the previous session. Each practical notebook therefore has the same bootstrap and participant configuration experience. The setup module is shared; the agent-building work is specific to this notebook.

## 2. Revisit the semantic catalogue: definitions before prompts

A catalogue is not a tool. It describes what business measures mean and which distinctions must be preserved. A tool is a permitted action that retrieves evidence. A Skill is a reusable procedure for analysing that evidence.

We will inspect the glossary and metric catalogue before constructing specialists.

In [ ]:
from pathlib import Path
import json

GLOSSARY_PATH = REPO_DIR / 'semantic_catalogue' / 'business_glossary.md'
METRICS_PATH = REPO_DIR / 'semantic_catalogue' / 'metrics.json'

glossary_text = GLOSSARY_PATH.read_text(encoding='utf-8')
metrics_catalogue = json.loads(METRICS_PATH.read_text(encoding='utf-8'))

print(glossary_text[:6500])
print("\nGlossary characters:", len(glossary_text))
print("Metric catalogue entries:", len(metrics_catalogue) if hasattr(metrics_catalogue, '__len__') else 'loaded')

### Three distinctions to keep intact

- **Inventory:** shortage units are calculated against safety stock using available quantity. Inventory value at risk is shortage units × unit cost; it is not lost revenue.
- **Supplier:** profile risk tier is a static supplier attribute. Observed delivery performance is calculated from delivered orders. One must not be used as a substitute for the other.
- **Fulfilment:** an overdue open order is an unresolved commitment whose promised date is before the snapshot date. A late delivered order is historical delivery performance.

These are semantic guardrails. A model prompt can repeat them, but the canonical definitions remain in the catalogue.

## 3. Build the governed Data Harness

The Data Harness is the controlled boundary between an agent and enterprise data sources. It is more than a collection of SQL tools: it resolves business metrics through the semantic catalogue, checks source and metric permissions, and delegates retrieval to a source adapter.

In this workshop, the BigQuery adapter is fully implemented. The same adapter contract can be implemented for additional governed sources later. The notebook does not provision a live cross-cloud federation service; it demonstrates the extensible federation pattern with a real BigQuery connection.

The layers are:

1. **Semantic catalogue:** what a metric means, its unit and caveats.
2. **Source adapter:** how to retrieve evidence from a particular platform.
3. **Harness policy:** which sources and metrics are allowed, plus read-only limits.
4. **Agent-facing result:** evidence accompanied by its metric definition and policy context.


In [ ]:
from data_harness.core import build_bigquery_harness

DATA_HARNESS = build_bigquery_harness(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    catalogue_path=REPO_DIR / 'semantic_catalogue' / 'metrics.json',
    location=BIGQUERY_LOCATION,
)

print('Registered source adapters:', list(DATA_HARNESS.adapters))
print('Allowed sources:', DATA_HARNESS.policy.allowed_sources)
print('Read-only:', DATA_HARNESS.policy.read_only)
print('Approved semantic metrics:', len(DATA_HARNESS.policy.allowed_metric_ids))


### Retrieve one metric through the harness

This call does not ask the agent to invent SQL or choose a table. It requests a known metric ID. The harness resolves its definition, checks that the source and metric are permitted, then calls the BigQuery adapter, which reads only its fixed semantic-view map.


In [ ]:
harness_evidence = DATA_HARNESS.retrieve(
    metric_id='inventory.shortage_units',
    source_name='bigquery',
    parameters={'limit': 5},
)

print('Metric contract:')
print(harness_evidence['metric'])
print('Governance envelope:')
print(harness_evidence['policy'])
print('Retrieved evidence:')
print(harness_evidence['evidence'])


### Data quality and freshness: check the evidence before reasoning

A Data Agent should not silently treat an old or incomplete snapshot as current. This lightweight preflight checks the latest approved inventory view for snapshot freshness and a few basic completeness signals. It is a workshop example—not a full data-quality framework—and uses the same configured BigQuery dataset.


In [ ]:
from datetime import date

FRESHNESS_THRESHOLD_DAYS = 30
health_sql = f'''
SELECT
  MAX(snapshot_date) AS latest_snapshot_date,
  COUNT(*) AS row_count,
  COUNTIF(product_id IS NULL) AS missing_product_ids,
  COUNTIF(warehouse_id IS NULL) AS missing_warehouse_ids,
  COUNTIF(snapshot_date IS NULL) AS missing_snapshot_dates
FROM `{PROJECT_ID}.{DATASET_ID}.v_inventory_product_risk`
'''
health_rows = list(bq.query(health_sql, location=BIGQUERY_LOCATION).result())
health = dict(health_rows[0].items())
latest_snapshot = health.get('latest_snapshot_date')
if latest_snapshot is None:
    freshness_status = 'UNKNOWN'
    snapshot_age_days = None
else:
    snapshot_age_days = (date.today() - latest_snapshot).days
    freshness_status = 'FRESH' if snapshot_age_days <= FRESHNESS_THRESHOLD_DAYS else 'STALE'

quality_status = 'PASS' if (
    health['row_count'] > 0
    and health['missing_product_ids'] == 0
    and health['missing_warehouse_ids'] == 0
    and health['missing_snapshot_dates'] == 0
) else 'REVIEW'
DATA_HEALTH_CONTEXT = {
    'freshness_status': freshness_status,
    'snapshot_age_days': snapshot_age_days,
    'freshness_threshold_days': FRESHNESS_THRESHOLD_DAYS,
    'quality_status': quality_status,
    **health,
}
print(json.dumps(DATA_HEALTH_CONTEXT, indent=2, default=str))
assert freshness_status in {'FRESH', 'STALE', 'UNKNOWN'}
assert quality_status in {'PASS', 'REVIEW'}


### What should the agent say when the data is stale?

The check above is a preflight signal. In a production design, pass this status into the agent's context or expose it as a dedicated read-only tool so the agent can qualify its answer. For this lab, inspect `DATA_HEALTH_CONTEXT` and practise writing a one-sentence caveat. Do not describe a stale snapshot as live inventory.


In [ ]:
if DATA_HEALTH_CONTEXT['freshness_status'] == 'STALE':
    freshness_caveat = (
        f"The latest inventory snapshot is {DATA_HEALTH_CONTEXT['snapshot_age_days']} days old; "
        "treat the analysis as a snapshot, not a live position."
    )
elif DATA_HEALTH_CONTEXT['freshness_status'] == 'UNKNOWN':
    freshness_caveat = "The inventory snapshot date could not be established; freshness is unknown."
else:
    freshness_caveat = "The inventory snapshot is within the configured freshness threshold."
print(freshness_caveat)


### Test the policy boundary

A governed harness should reject an unapproved source before attempting a connection. This is a simple policy check, not a substitute for IAM, BigQuery permissions, network controls or source-native security.


In [ ]:
try:
    DATA_HARNESS.retrieve(
        metric_id='inventory.shortage_units',
        source_name='unapproved_external_source',
    )
except PermissionError as exc:
    print('Expected policy rejection:', exc)
else:
    raise AssertionError('An unapproved source was not rejected.')


### Plan a business question using semantic contracts

The Harness should not merely guard a function call. It should help resolve the business question into a small set of approved metrics before retrieval. The planner uses catalogue metadata and conservative phrase matching; unfamiliar questions trigger clarification. In a production implementation, a model could propose a plan, but the same deterministic policy checks must validate it before execution.


In [ ]:
from data_harness.core import plan_business_question, SemanticCatalogue

PLANNING_EXAMPLES = [
    "Which suppliers have the weakest on-time delivery performance?",
    "Which products are below safety stock and what is their value at risk?",
    "Which suppliers are associated with inventory shortages and overdue orders?",
]
for question in PLANNING_EXAMPLES:
    plan = plan_business_question(question)
    print("\nQUESTION:", question)
    print("DOMAINS:", plan["domains"])
    print("METRICS:", plan["metric_ids"])
    print("DIMENSIONS:", plan["dimensions"])


### Discover the schemas behind the semantic contracts

The glossary explains business meaning; schema discovery checks the physical columns that implement it. Inspect the approved supplier, inventory and fulfilment views and compare their keys and date fields. This is how a Data Agent grounds a proposed analysis in both semantic definitions and the real warehouse schema.

In [ ]:
from data_harness.governance import BigQuerySchemaDiscovery

APPROVED_VIEWS = (
    'v_inventory_product_risk', 'v_inventory_risk_summary',
    'v_supplier_delivery_performance', 'v_fulfillment_order_exposure',
    'v_fulfillment_risk_summary', 'v_supplier_cross_domain_risk',
)
schema_discovery = BigQuerySchemaDiscovery(workshop.bigquery, PROJECT_ID, DATASET_ID, APPROVED_VIEWS)
for view_name in ('v_inventory_product_risk', 'v_supplier_delivery_performance', 'v_fulfillment_order_exposure', 'v_supplier_cross_domain_risk'):
    snapshot = schema_discovery.discover(view_name)
    print('\n', view_name)
    print(', '.join(f"{col.name}:{col.field_type}" for col in snapshot.columns))


### Inspect query governance and cost controls

A semantic plan is not permission to execute arbitrary SQL. The query validator checks that a proposed statement reads only approved semantic views, is read-only, contains one statement and has a bounded result. The execution helper performs a dry run and applies a maximum-bytes-billed ceiling before running it.

The same principle applies to query cost: keep queries on approved semantic views, require bounded results, use dry runs where appropriate, and set `maximum_bytes_billed`. The first notebook demonstrates the dry-run and bytes-billed ceiling in an executable query. Here we focus on the allow-list and row-bound policy used by the agent tools.


In [ ]:
from data_harness.governance import ReadOnlyQueryValidator

query_validator = ReadOnlyQueryValidator(PROJECT_ID, DATASET_ID, APPROVED_VIEWS)
allowed = f"SELECT supplier_id, supplier_name FROM `{PROJECT_ID}.{DATASET_ID}.v_supplier_delivery_performance` LIMIT 10"
blocked = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.suppliers` LIMIT 10"
print('Allowed:', query_validator.validate(allowed))
print('Blocked:', query_validator.validate(blocked))
assert query_validator.validate(allowed).approved
assert not query_validator.validate(blocked).approved


### Understand why grain matters

Cross-domain analysis can silently produce incorrect totals if raw inventory rows are joined directly to raw order rows. Inventory is at product/warehouse/snapshot grain; orders are at order grain. The correct pattern is to aggregate each domain to a common supplier grain first, then join the aggregates. Notebook 3 uses this pattern in a governed BigQuery view.


### How this becomes federated

To connect another platform, implement the `DataSourceAdapter` contract and register the adapter under a source name. Then explicitly allow that source and only the metric IDs it is approved to serve. The additional platform still needs its own credentials, network path, IAM/ACLs, audit controls and integration tests. The agent never receives unrestricted connection details or arbitrary SQL.

**Concepts practised:** Data Harness, semantic grounding, tool engineering and Harness Engineering.


## 4. Agent Skills: package repeatable domain expertise

An Agent Skill is a reusable package of domain-specific instructions, workflow steps, response structure and guardrails. It complements the business glossary:

- the **catalogue** defines terms and metrics;
- the **tool** retrieves a permitted slice of evidence;
- the **Skill** tells the agent how to analyse that evidence consistently.

Inspect the Skill package files. Notice that each has a focused purpose and a repeatable analysis workflow.

In [ ]:
SKILLS_DIR = REPO_DIR / 'skills'
SKILL_FILES = {
    'inventory': SKILLS_DIR / 'inventory-risk-analysis' / 'SKILL.md',
    'supplier': SKILLS_DIR / 'supplier-performance-analysis' / 'SKILL.md',
    'fulfilment': SKILLS_DIR / 'fulfilment-exposure-analysis' / 'SKILL.md',
}

skill_text = {name: path.read_text(encoding='utf-8') for name, path in SKILL_FILES.items()}
for name, text in skill_text.items():
    print(f"\n{'='*18} {name.upper()} SKILL {'='*18}")
    print(text[:1800])

### Quick design review

For each Skill, identify:

1. the business questions it covers;
2. the evidence it expects from tools;
3. the metric distinctions it must preserve;
4. the actions it must never take.

**Pause (1 minute):** which parts belong in the semantic catalogue, and which parts belong in the Skill? A metric formula belongs in the catalogue; the ordered procedure for comparing suppliers belongs in the Skill.

## 5. Build a narrow, read-only tool surface for each domain

We will create one shared BigQuery tool object, then expose only selected functions to each specialist. This is an application-level allow-list: the supplier specialist will not receive inventory or fulfilment functions. Skills guide behaviour but do not replace IAM or data permissions.

In [ ]:
from functools import wraps
from tools.bigquery_tools import SupplyChainBigQueryTools

supply_chain_tools = SupplyChainBigQueryTools(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    location=BIGQUERY_LOCATION,
)

# Every agent-facing callable passes through the same semantic and policy gate.
def governed(metric_id, tool_fn):
    @wraps(tool_fn)
    def call(**kwargs):
        return DATA_HARNESS.invoke_tool(metric_id, tool_fn, kwargs, source_name='bigquery')
    call.__doc__ = (tool_fn.__doc__ or '') + ' Governed by the workshop Data Harness; read-only.'
    return call

# Inventory domain

def get_inventory_risk(limit: int = 10):
    """Read product-level inventory risk from the approved semantic view."""
    return supply_chain_tools.inventory_risk(inventory_status=None, category=None, limit=limit)

def get_inventory_summary():
    """Read canonical inventory portfolio metrics for the snapshot."""
    return supply_chain_tools.inventory_summary()

# Supplier domain

def get_supplier_performance(limit: int = 20):
    """Read supplier profile tier and observed delivery performance."""
    return supply_chain_tools.supplier_performance(supplier_risk_tier=None, limit=limit)

# Fulfilment domain

def get_fulfillment_exposure(limit: int = 20):
    """Read overdue open-order exposure from the approved semantic view."""
    return supply_chain_tools.fulfillment_exposure(order_status=None, only_overdue=True, limit=limit)

def get_fulfillment_summary():
    """Read canonical fulfilment portfolio metrics for the snapshot."""
    return supply_chain_tools.fulfillment_summary()

ALL_TOOL_FUNCTIONS = {
    'get_inventory_risk': governed('inventory.value_at_risk_gbp', get_inventory_risk),
    'get_inventory_summary': governed('inventory.shortage_units', get_inventory_summary),
    'get_supplier_performance': governed('supplier.on_time_delivery_rate', get_supplier_performance),
    'get_fulfillment_exposure': governed('fulfillment.overdue_open_orders', get_fulfillment_exposure),
    'get_fulfillment_summary': governed('fulfillment.delayed_open_order_value_gbp', get_fulfillment_summary),
}

print('Available tool functions:', sorted(ALL_TOOL_FUNCTIONS))

### Inspect the tool contracts

A specialist should receive the smallest toolset that can answer its assigned questions. The signatures and docstrings are part of the tool contract presented to the model. They should be understandable without reading the implementation.

In [ ]:
import inspect

for tool_name, tool_fn in ALL_TOOL_FUNCTIONS.items():
    print(f"\n{tool_name}{inspect.signature(tool_fn)}")
    print(inspect.getdoc(tool_fn))

### Execute tools independently before involving an LLM

This is an important debugging boundary. If a query fails here, the problem is in data access, permissions or the tool. If the tool works here but an agent answer is poor, investigate context, Skill, tool selection or model behaviour.

In [ ]:
inventory_check = get_inventory_risk(limit=5)
supplier_check = get_supplier_performance(limit=5)
fulfilment_check = get_fulfillment_exposure(limit=5)

for label, result in [
    ('Inventory', inventory_check),
    ('Supplier', supplier_check),
    ('Fulfilment', fulfilment_check),
]:
    print(f"{label}: rows={result['row_count']} read_only={result['read_only']}")
    if result['rows']:
        print('  Example fields:', sorted(result['rows'][0].keys()))

### Tool allow-lists are explicit

Create a mapping that documents the intended permissions. This mapping is useful for inspection and later tests. It is not a substitute for Google Cloud IAM: the underlying identity must still have only the permissions it needs.

In [ ]:
SPECIALIST_TOOL_ALLOWLISTS = {
    'inventory_specialist': ['get_inventory_risk', 'get_inventory_summary'],
    'supplier_specialist': ['get_supplier_performance'],
    'fulfilment_specialist': ['get_fulfillment_exposure', 'get_fulfillment_summary'],
}

SPECIALIST_TOOLS = {
    specialist: [ALL_TOOL_FUNCTIONS[name] for name in permitted]
    for specialist, permitted in SPECIALIST_TOOL_ALLOWLISTS.items()
}

for specialist, tools in SPECIALIST_TOOLS.items():
    print(f"{specialist}: {[tool.__name__ for tool in tools]}")

### Test the boundary before building agents

The following checks make sure each specialist has exactly the expected tool names. They are simple design-time checks, not a cloud security certification.

In [ ]:
expected_tool_names = {
    'inventory_specialist': {'get_inventory_risk', 'get_inventory_summary'},
    'supplier_specialist': {'get_supplier_performance'},
    'fulfilment_specialist': {'get_fulfillment_exposure', 'get_fulfillment_summary'},
}

for specialist, tools in SPECIALIST_TOOLS.items():
    actual = {tool.__name__ for tool in tools}
    assert actual == expected_tool_names[specialist], (specialist, actual)
    print(f"PASS {specialist}: {sorted(actual)}")

assert 'get_inventory_risk' not in expected_tool_names['supplier_specialist']
assert 'get_supplier_performance' not in expected_tool_names['inventory_specialist']
print('All specialist tool-boundary checks passed.')

## 6. Load the Skills into ADK

The next cell uses the ADK Skill API exposed by the pinned workshop dependency. It loads each Skill from its directory and wraps it in a `SkillToolset`. If the installed ADK build does not expose that API, the cell stops with a clear error rather than silently claiming that the Skill was attached.

In [ ]:
from google.adk.skills import load_skill_from_dir
from google.adk.tools.skill_toolset import SkillToolset

ADK_SKILL_TOOLSETS = {}
for skill_name, skill_dir in [
    ('inventory', SKILLS_DIR / 'inventory-risk-analysis'),
    ('supplier', SKILLS_DIR / 'supplier-performance-analysis'),
    ('fulfilment', SKILLS_DIR / 'fulfilment-exposure-analysis'),
]:
    loaded_skill = load_skill_from_dir(skill_dir)
    ADK_SKILL_TOOLSETS[skill_name] = SkillToolset(skills=[loaded_skill])
    print(f"Loaded ADK Skill: {skill_name} from {skill_dir.name}")

print('ADK Skill toolsets ready:', sorted(ADK_SKILL_TOOLSETS))

### Skill attachment is not a security boundary

A Skill provides task guidance and structured domain knowledge. It does not grant or revoke Google Cloud access. The agent's tool list, application policy and the identity's IAM permissions determine what data can actually be read. We deliberately keep all workshop tools read-only.

## 7. Define the three specialist contracts

Each specialist gets:

- a narrow job description and instruction set;
- its domain's read-only tool allow-list;
- its corresponding Agent Skill;
- the same snapshot and warehouse context, with domain-specific metric rules.

The agent names are stable identifiers used later by the coordinator in Notebook 3.

In [ ]:
INVENTORY_SPECIALIST_INSTRUCTION = """
You are the Inventory Specialist for Essex Central Distribution Centre.
Answer only questions about available stock, safety-stock gaps, stockouts,
shortage units and inventory value at risk. Use only your supplied inventory
read-only tools and the Inventory Risk Analysis Skill.

Definitions: available quantity = on-hand minus reserved; shortage units =
max(safety stock minus available quantity, 0); inventory value at risk =
shortage units multiplied by unit cost in GBP. This is not lost revenue.

State snapshot date 2026-10-01 and warehouse WH-ESSEX-01. Use the summary tool
for portfolio totals and product-level tool for examples. Do not invent causes,
trends or missing values. Never mutate inventory or place orders.
"""

SUPPLIER_SPECIALIST_INSTRUCTION = """
You are the Supplier Specialist for Essex Central Distribution Centre.
Answer only questions about supplier profile risk tier, supplier-level standard
lead time and observed delivery performance. Use only your supplied supplier
read-only tool and Supplier Performance Analysis Skill.

Keep supplier profile risk tier separate from observed delivery performance.
On-time delivery rate uses delivered orders with actual delivery dates; open and
cancelled orders are excluded. Show the denominator when available. Lead time
is supplier-level, not product-level. State snapshot date 2026-10-01 and warehouse
WH-ESSEX-01. Do not infer causality or trends from one snapshot. Never contact
suppliers or amend contracts.
"""

FULFILMENT_SPECIALIST_INSTRUCTION = """
You are the Fulfilment Specialist for Essex Central Distribution Centre.
Answer only questions about open orders, overdue open commitments, current
 overdue days and delayed open-order value. Use only your supplied fulfilment
read-only tools and Fulfilment Exposure Analysis Skill.

An overdue open order is an unresolved order whose promised date is before the
snapshot date. Keep late delivered orders separate from overdue open orders.
Delayed open-order value is an exposure indicator, not realised loss or lost
revenue. State snapshot date 2026-10-01 and warehouse WH-ESSEX-01. Never amend
orders, promise dates or contact customers or suppliers.
"""

# Inject the preflight freshness signal into the inventory specialist's context.
INVENTORY_SPECIALIST_INSTRUCTION += (
    "\n\nData quality and freshness preflight: "
    f"freshness={DATA_HEALTH_CONTEXT['freshness_status']}; "
    f"snapshot_age_days={DATA_HEALTH_CONTEXT['snapshot_age_days']}; "
    f"quality={DATA_HEALTH_CONTEXT['quality_status']}. "
    "If stale or unknown, explicitly qualify every inventory answer and do not describe it as live data."
)

SPECIALIST_INSTRUCTIONS = {
    'inventory_specialist': INVENTORY_SPECIALIST_INSTRUCTION,
    'supplier_specialist': SUPPLIER_SPECIALIST_INSTRUCTION,
    'fulfilment_specialist': FULFILMENT_SPECIALIST_INSTRUCTION,
}

### Build the specialists

We use ADK `Agent`, `App` and `Runner`, the same execution pattern introduced in Notebook 1. Each specialist has its own app, session service and runner so its execution can be tested independently. This is still a single-agent-per-question pattern; the coordinator and delegation logic are built in Notebook 3.

In [ ]:
from google.adk.agents import Agent
from google.adk.apps import App
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

SPECIALIST_CONFIG = {
    'inventory_specialist': {
        'description': 'Analyses inventory availability, safety-stock gaps and inventory value at risk.',
        'tools': SPECIALIST_TOOLS['inventory_specialist'] + [ADK_SKILL_TOOLSETS['inventory']],
    },
    'supplier_specialist': {
        'description': 'Analyses supplier profile risk and observed delivery performance.',
        'tools': SPECIALIST_TOOLS['supplier_specialist'] + [ADK_SKILL_TOOLSETS['supplier']],
    },
    'fulfilment_specialist': {
        'description': 'Analyses overdue open commitments and fulfilment exposure.',
        'tools': SPECIALIST_TOOLS['fulfilment_specialist'] + [ADK_SKILL_TOOLSETS['fulfilment']],
    },
}

SPECIALIST_AGENTS = {}
SPECIALIST_RUNNERS = {}
SPECIALIST_SESSIONS = {}

for specialist_name, spec in SPECIALIST_CONFIG.items():
    agent = Agent(
        name=specialist_name,
        model=MODEL_ID,
        description=spec['description'],
        instruction=SPECIALIST_INSTRUCTIONS[specialist_name],
        tools=spec['tools'],
    )
    app = App(name=f"gdg_essex_{specialist_name}", root_agent=agent)
    session_service = InMemorySessionService()
    runner = Runner(app=app, session_service=session_service)
    SPECIALIST_AGENTS[specialist_name] = agent
    SPECIALIST_RUNNERS[specialist_name] = runner
    SPECIALIST_SESSIONS[specialist_name] = session_service
    print(f"Created {specialist_name}: {agent.name}")

### Review the specialist design

Check the separation between the three agents. A supplier question should not be answered by the inventory specialist simply because it has access to all the data. Domain boundaries reduce irrelevant context, simplify evaluation and make tool access easier to review.

In [ ]:
for specialist_name, agent in SPECIALIST_AGENTS.items():
    print(f"\n{specialist_name}")
    print('  Description:', agent.description)
    print('  Tool allow-list:', SPECIALIST_TOOL_ALLOWLISTS[specialist_name])
    print('  Skill attached:', True)
    print('  Read-only instruction:', 'read-only' in SPECIALIST_INSTRUCTIONS[specialist_name].lower())

## 8. Run each specialist independently

We now ask one focused question per specialist. The response should be grounded in that specialist's tools and Skill. The precise wording and event sequence can vary, but the business distinctions should remain stable.

In [ ]:
async def ask_specialist(specialist_name: str, question: str, user_id: str = 'gdg_essex_participant'):
    runner = SPECIALIST_RUNNERS[specialist_name]
    app_name = f"gdg_essex_{specialist_name}"
    session = await SPECIALIST_SESSIONS[specialist_name].create_session(
        app_name=app_name,
        user_id=user_id,
    )
    message = genai_types.Content(
        role='user',
        parts=[genai_types.Part(text=question)],
    )
    answer_parts = []
    event_log = []
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=message,
    ):
        event_log.append({
            'author': getattr(event, 'author', None),
            'is_final': event.is_final_response(),
            'has_content': bool(getattr(event, 'content', None)),
        })
        if event.is_final_response() and event.content and event.content.parts:
            answer_parts.extend(
                part.text for part in event.content.parts
                if getattr(part, 'text', None)
            )
    return {
        'answer': '\n'.join(answer_parts).strip(),
        'events': event_log,
        'session_id': session.id,
    }

print('Specialist runner helper is ready.')

### Inventory Specialist: analyse stock exposure

Ask for the five highest-value inventory gaps and the portfolio context. Look for the distinction between shortage units and available quantity, and check that the agent does not describe the cost value as lost revenue.

In [ ]:
INVENTORY_QUESTION = (
    'Which five products have the highest inventory value at risk? '
    'Show shortage units and GBP value at risk, then explain the overall '
    'inventory position using the portfolio summary. State the snapshot date.'
)
inventory_specialist_result = await ask_specialist('inventory_specialist', INVENTORY_QUESTION)
print(inventory_specialist_result['answer'])

### Supplier Specialist: separate profile from performance

Ask for the suppliers with the weakest observed on-time delivery performance. The agent should distinguish the static supplier profile tier from delivery outcomes, and show the delivered-order denominator if available.

In [ ]:
SUPPLIER_QUESTION = (
    'Which suppliers have the weakest observed on-time delivery performance? '
    'Compare the delivery rate and late deliveries with each supplier profile '
    'risk tier. Explain why those are different measures.'
)
supplier_specialist_result = await ask_specialist('supplier_specialist', SUPPLIER_QUESTION)
print(supplier_specialist_result['answer'])

### Fulfilment Specialist: focus on unresolved commitments

Ask for the most overdue open orders and current exposure. The specialist should not mix overdue open orders with orders that were delivered late in the past.

In [ ]:
FULFILMENT_QUESTION = (
    'Which open fulfilment orders are most overdue? Summarise current overdue '
    'days and delayed open-order value, and explain how this differs from '
    'late delivered orders.'
)
fulfilment_specialist_result = await ask_specialist('fulfilment_specialist', FULFILMENT_QUESTION)
print(fulfilment_specialist_result['answer'])

### Compare the evidence, not just the prose

A specialist's natural-language response is not the canonical data source. Review the tool events and the deterministic tool outputs when validating important numbers. The table below records the execution facts captured by this notebook.

In [ ]:
SPECIALIST_RESULTS = {
    'inventory_specialist': inventory_specialist_result,
    'supplier_specialist': supplier_specialist_result,
    'fulfilment_specialist': fulfilment_specialist_result,
}

for name, result in SPECIALIST_RESULTS.items():
    print(f"{name}: events={len(result['events'])}, final_answer={bool(result['answer'])}")
    print('  event authors:', sorted({str(e['author']) for e in result['events']}))

## 9. Add a deterministic routing preview

Before introducing the coordinator in Notebook 3, we can test the idea of specialist selection. A router classifies the question into a domain route; a policy then determines which specialist is eligible. The core path uses deterministic rules, so no optional provider key is required.

In [ ]:
from orchestration.router import route_question
from orchestration.policy import ROUTE_TOOLS

ROUTE_TO_SPECIALIST = {
    'INVENTORY_RISK': 'inventory_specialist',
    'SUPPLIER_RISK': 'supplier_specialist',
    'FULFILMENT_DELAY': 'fulfilment_specialist',
    'GENERAL_QUESTION': 'coordinator_or_clarification',
    'CROSS_DOMAIN': 'multiple_specialists',
    'UNCERTAIN': 'ask_for_clarification',
}

routing_examples = [
    'Which products are below safety stock?',
    'Which suppliers have the poorest on-time delivery?',
    'Which open orders are overdue?',
    'Compare supplier delivery performance with overdue fulfilment exposure.',
    'What should I do about the business?',
]

for question in routing_examples:
    decision = route_question(question, use_decision_model=False)
    print(f"\nQuestion: {question}")
    print(f"  Route: {decision.route} | confidence={decision.confidence:.2f} | source={decision.source}")
    print(f"  Next: {ROUTE_TO_SPECIALIST.get(decision.route, 'review route')}")

### Inspect the route-to-tool policy

The policy is a separate layer from the model's reasoning. It makes the intended mapping inspectable and testable. For single-domain routes, the policy lists the allowed tools; cross-domain and uncertain routes do not receive a broad direct toolset.

In [ ]:
for route_name, permitted_tools in ROUTE_TOOLS.items():
    print(f"{route_name:18s} -> {list(permitted_tools)}")

assert ROUTE_TOOLS['SUPPLIER_RISK'] == ('get_supplier_performance',)
assert 'get_inventory_risk' not in ROUTE_TOOLS['SUPPLIER_RISK']
assert ROUTE_TOOLS['CROSS_DOMAIN'] == ()
print('Routing policy assertions passed.')

### Optional structured routing

The repository includes an optional Decision 1 adapter for a structured model-based routing experiment. It is intentionally not enabled in the core lab: the provider identity, endpoint, authentication and availability are environment-specific. Notebook 3 can compare this optional path with the deterministic baseline. The workshop still completes when the optional integration is unavailable.

## 10. Knowledge check and hands-on challenge

Answer these before moving to Notebook 3:

1. What is the difference between a semantic catalogue, a tool and an Agent Skill?
2. Why does the supplier specialist not receive inventory tools?
3. Why is a Skill not a security boundary?
4. What is the difference between supplier profile risk tier and observed delivery performance?
5. Why should an uncertain route avoid receiving a broad toolset?

**Core challenge (3 minutes):** add one question of your own to the routing examples. Predict the route, run the deterministic router, and explain whether the selected specialist has enough evidence and the right tools to answer it.

**Explore further:** inspect one Skill file and propose one additional validation step that would make its workflow more reliable. Do not add write actions.

In [ ]:
# Participant challenge — edit only the question text and run this cell.
MY_ROUTING_QUESTION = 'Compare inventory shortages with supplier delivery performance.'
my_decision = route_question(MY_ROUTING_QUESTION, use_decision_model=False)
print('Question:', MY_ROUTING_QUESTION)
print('Route:', my_decision.route)
print('Reason:', my_decision.reason)
print('Suggested handling:', ROUTE_TO_SPECIALIST.get(my_decision.route, 'Review manually'))

## Where this notebook fits in the 10 workshop concepts

Notebook 2 deepens the concepts introduced in the opening talk and Notebook 1. The coverage below describes what this lab actually demonstrates; it does not claim that every enterprise capability is fully implemented.

| # | Workshop concept | Notebook 2 contribution |
|---:|---|---|
| 1 | Agent mental model | Extends one agent into three bounded specialists |
| 2 | Harness Engineering | Reuses ADK apps, runners, sessions and event capture |
| 3 | Context Engineering | Separates shared business context from domain-specific instructions |
| 4 | Data Harness | Implements a policy-enforcing source-adapter layer, semantic metric resolution and read-only BigQuery retrieval |
| 5 | Semantic grounding | Uses the shared glossary and canonical metric catalogue |
| 6 | Tool engineering | Defines tool contracts, tests them independently and applies specialist allow-lists |
| 7 | Agent Skills | Loads reusable Inventory, Supplier and Fulfilment Skills through ADK SkillToolset |
| 8 | Routing and specialisation | Runs deterministic route classification and maps routes to specialists |
| 9 | Orchestration and autonomy | Prepares specialist boundaries for coordinator delegation in Notebook 3 |
| 10 | Evaluation and observability | Checks tool results, specialist answers, event authors and routing policy |

### What comes next

Notebook 3 builds the coordinator that selects one specialist or delegates to multiple specialists for a cross-domain question. It adds bounded delegation, synthesis, failure handling, evaluation and trace review. Keep this notebook's specialist contracts and tool boundaries as the foundation.